# RAG: Baseline (Bi-Encoder) vs. Re-Ranking (Cross-Encoder)

**Document:** FIDE Laws of Chess (official FIDE Handbook E.I.01, 25 pages). A rulebook with lots of general rules, exceptions ("except when…"), penalties and prohibitions, which makes it a good stress test for retrieval.

**Pipelines** (both run over the *exact same* chunk corpus and the *same* LLM prompt):

| | Pipeline 1 – Baseline | Pipeline 2 – Re-Ranking |
|---|---|---|
| Stage 1 | Bi-encoder `all-MiniLM-L6-v2` + FAISS (cosine), Top-10 | same Top-10 candidates |
| Stage 2 | take Top-3 by cosine | score all 10 (query, chunk) pairs with `cross-encoder/ms-marco-MiniLM-L-6-v2`, re-sort, take Top-3 |
| Generation | LLM → **Answer 1** | LLM → **Answer 2** |

**LLM:** `Qwen/Qwen2.5-3B-Instruct` run locally (Hugging Face `transformers`, greedy decoding). If a `GEMINI_API_KEY` environment variable is set, the notebook uses Gemini instead.

Notebook outline:
1. Setup & configuration
2. Load and clean the PDF
3. Chunking
4. Pipeline 1: bi-encoder index (FAISS)
5. Pipeline 2: cross-encoder re-ranker
6. LLM + shared prompt
7. Failure-mode stress tests (rank shifts + Answer 1 vs Answer 2)
8. Cases where re-ranking does **not** help
9. Aggregate retrieval metrics on a wider labelled query set
10. Latency profiling
11. Scaling experiment (why not cross-encode the whole corpus?)

## 1. Setup & configuration

In [1]:
# If running somewhere fresh (e.g. Colab), install the dependencies first:
# !pip install torch sentence-transformers transformers accelerate faiss-cpu pypdf pandas google-genai

import os, re, time, json, textwrap, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import faiss
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder

warnings.filterwarnings("ignore")
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

# ---- Configuration -------------------------------------------------------
PDF_PATH        = Path("data/FIDE_Laws_of_Chess.pdf")
RESULTS_DIR     = Path("results"); RESULTS_DIR.mkdir(exist_ok=True)

BI_ENCODER_NAME    = "sentence-transformers/all-MiniLM-L6-v2"   # dense bi-encoder
CROSS_ENCODER_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"     # re-ranker
LOCAL_LLM_NAME     = "Qwen/Qwen2.5-3B-Instruct"                 # used when no Gemini key
GEMINI_MODEL_NAME  = "gemini-2.5-flash"

CHUNK_SIZE    = 500   # max characters per chunk
CHUNK_OVERLAP = 100   # characters carried over from the previous chunk
TOP_K_RETRIEVE = 10   # candidates from stage 1
TOP_N_CONTEXT  = 3    # chunks given to the LLM

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0); np.random.seed(0)
print("Device:", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")

Device: cuda | NVIDIA GeForce RTX 4060 Laptop GPU


## 2. Load and clean the PDF
`pypdf` extracts text page by page. Cleaning steps:
* drop lines that contain only a page number
* normalise curly quotes and collapse repeated whitespace

The PDF has a few artefacts from extraction (for example `horizon tal` and `leg al`). I left them in on purpose, because real-world PDFs are messy and both pipelines see exactly the same text.

In [2]:
def load_pdf_pages(path):
    # Return a list of (page_number, cleaned_text).
    reader = PdfReader(str(path))
    pages = []
    for i, page in enumerate(reader.pages):
        t = page.extract_text() or ""
        t = re.sub(r"^\s*\d+\s*$", "", t, flags=re.M)           # page-number-only lines
        t = t.replace("’", "'").replace("‘", "'")      # curly quotes -> straight
        t = re.sub(r"[ \t]+", " ", t)                            # collapse spaces
        t = re.sub(r"\s*\n\s*", "\n", t).strip()                 # collapse blank lines
        pages.append((i + 1, t))
    return pages

pages = load_pdf_pages(PDF_PATH)
print(f"Pages: {len(pages)} | total characters: {sum(len(t) for _, t in pages):,}")
print("\n--- Sample (page 7) ---\n", pages[6][1][:800])

Pages: 25 | total characters: 52,566

--- Sample (page 7) ---
 b. (1) The right to castle has been lost:
[a] if the king has already moved, or
[b] with a rook that has already moved.
(2) Castling is prevented temporarily:
[a] if the square on which the king stands, or the square which it must
cross, or the square which it is to occupy, is attacked by one or more of
the opponent's pieces, or
[b] if there is any piece between the king and the rook with which castling is
to be effected.
3.9 The king is said to be 'in check' if it is attacked by one or more of the opponent's pieces,
even if such pieces are constrained from movin g to that square because they would then
leave or place their own king in check. No piece can be moved that will either expose the
king of the same colour to check or leave that king in check.
Article 4: The act of moving the piec


## 3. Chunking
I use sentence-aware fixed-size chunking:
* Each page is split into sentences on `.`, `;` and `:`. Many clauses in the rulebook end with `;` or `:`.
* Sentences are packed into chunks of at most `CHUNK_SIZE` characters.
* The last ~`CHUNK_OVERLAP` characters of a chunk are repeated at the start of the next one, so a rule that crosses a boundary isn't lost.
* Page 1 (the table of contents) is skipped. It only contains article titles, which would act as distractors.

Each chunk keeps its page number so it can be traced back to the document.

In [3]:
def split_sentences(text):
    text = text.replace("\n", " ")
    return [s for s in re.split(r"(?<=[.;:])\s+", text) if s.strip()]

def chunk_pages(pages, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP, skip_pages=(1,)):
    chunks = []
    for page_no, text in pages:
        if page_no in skip_pages:
            continue
        current = ""
        for sent in split_sentences(text):
            if current and len(current) + len(sent) > chunk_size:
                chunks.append({"page": page_no, "text": current.strip()})
                # carry the tail of the previous chunk (cut at a word boundary) as overlap
                tail = current[-overlap:] if overlap else ""
                current = tail[tail.find(" ") + 1:] if overlap else ""
            current += " " + sent
        if current.strip():
            chunks.append({"page": page_no, "text": current.strip()})
    for i, c in enumerate(chunks):
        c["chunk_id"] = i
    return chunks

chunks = chunk_pages(pages)
chunk_texts = [c["text"] for c in chunks]
lens = np.array([len(t) for t in chunk_texts])
print(f"Chunks: {len(chunks)} | chars per chunk: mean={lens.mean():.0f}, min={lens.min()}, max={lens.max()}")
pd.DataFrame(chunks[20:24])[["chunk_id", "page", "text"]]

Chunks: 156 | chars per chunk: mean=412, min=111, max=747


,chunk_id,page,text
0,20,6,"on its original square , then that rook is transferred to the square the king has just crossed. Before white kingsid..."
1,21,7,"b. (1) The right to castle has been lost: [a] if the king has already moved, or [b] with a rook that has already mov..."
2,22,7,or [b] if there is any piece between the king and the rook with which castling is to be effected. 3.9 The king is sa...
3,23,7,will either expose the king of the same colour to check or leave that king in check. Article 4: The act of moving th...


## 4. Pipeline 1: bi-encoder index (FAISS)
The bi-encoder embeds every chunk **once, offline**. Embeddings are L2-normalised, so the inner product in a FAISS `IndexFlatIP` equals cosine similarity. At query time only the query is embedded, followed by a nearest-neighbour search.

In [4]:
bi_encoder = SentenceTransformer(BI_ENCODER_NAME, device=DEVICE)

t0 = time.perf_counter()
chunk_emb = bi_encoder.encode(chunk_texts, batch_size=64, normalize_embeddings=True,
                              convert_to_numpy=True, show_progress_bar=False).astype("float32")
index_time = time.perf_counter() - t0

index = faiss.IndexFlatIP(chunk_emb.shape[1])   # exact inner-product search
index.add(chunk_emb)
print(f"Indexed {index.ntotal} chunks, dim={chunk_emb.shape[1]}, offline indexing time={index_time:.2f}s")

def baseline_retrieve(query, k=TOP_K_RETRIEVE):
    # Stage 1: embed query, return top-k (chunk_ids, cosine scores).
    q = bi_encoder.encode([query], normalize_embeddings=True, convert_to_numpy=True).astype("float32")
    scores, ids = index.search(q, k)
    return ids[0].tolist(), scores[0].tolist()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Indexed 156 chunks, dim=384, offline indexing time=0.67s


## 5. Pipeline 2: cross-encoder re-ranker
The cross-encoder reads the query and the chunk **together** in one forward pass (`[CLS] query [SEP] chunk [SEP]`). Every query token can attend to every chunk token, and the model outputs a single relevance logit. Nothing can be pre-computed, so it is applied only to the 10 candidates from stage 1.

In [5]:
cross_encoder = CrossEncoder(CROSS_ENCODER_NAME, device=DEVICE)

def rerank(query, candidate_ids):
    # Stage 2: score every (query, chunk) pair and sort descending by cross-encoder score.
    pairs = [(query, chunk_texts[i]) for i in candidate_ids]
    ce_scores = cross_encoder.predict(pairs, batch_size=16, show_progress_bar=False)
    order = np.argsort(-ce_scores)
    return [candidate_ids[j] for j in order], [float(ce_scores[j]) for j in order]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

## 6. LLM + shared prompt
Both pipelines use the **identical** prompt template. Only the 3 context chunks differ. Decoding is greedy (`do_sample=False`), so the answers are deterministic and the comparison is fair.

In [6]:
SYSTEM_PROMPT = (
    "You are an assistant that answers questions about the FIDE Laws of Chess. "
    "Answer ONLY using the context passages provided. If the context does not contain "
    "the answer, say \"The provided context does not contain this information.\" "
    "Answer in 2-4 sentences."
)

def build_prompt(query, ctx_ids):
    context = "\n\n".join(f"[Passage {n+1} | page {chunks[i]['page']}]\n{chunk_texts[i]}"
                          for n, i in enumerate(ctx_ids))
    return f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"

USE_GEMINI = bool(os.environ.get("GEMINI_API_KEY"))

if USE_GEMINI:
    from google import genai
    from google.genai import types
    gclient = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
    LLM_NAME = GEMINI_MODEL_NAME

    def llm_generate(user_prompt):
        resp = gclient.models.generate_content(
            model=GEMINI_MODEL_NAME, contents=user_prompt,
            config=types.GenerateContentConfig(system_instruction=SYSTEM_PROMPT, temperature=0.0))
        return resp.text.strip()
else:
    from transformers import AutoTokenizer, AutoModelForCausalLM
    LLM_NAME = LOCAL_LLM_NAME
    tok = AutoTokenizer.from_pretrained(LOCAL_LLM_NAME)
    llm = AutoModelForCausalLM.from_pretrained(
        LOCAL_LLM_NAME, torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
        device_map=DEVICE)
    llm.eval()

    @torch.no_grad()
    def llm_generate(user_prompt, max_new_tokens=200):
        messages = [{"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user", "content": user_prompt}]
        inputs = tok.apply_chat_template(messages, add_generation_prompt=True,
                                         return_tensors="pt", return_dict=True).to(llm.device)
        out = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                           pad_token_id=tok.eos_token_id)
        return tok.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print("LLM:", LLM_NAME)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

LLM: Qwen/Qwen2.5-3B-Instruct


### The two end-to-end pipelines

In [7]:
def pipeline_1(query):
    # Baseline: bi-encoder top-10 -> take top-3 directly -> LLM (Answer 1).
    ids, cos = baseline_retrieve(query)
    ctx = ids[:TOP_N_CONTEXT]
    return {"top10": ids, "cos": cos, "context": ctx, "answer": llm_generate(build_prompt(query, ctx))}

def pipeline_2(query, top10=None):
    # Re-ranking: same top-10 -> cross-encoder re-sort -> top-3 -> LLM (Answer 2).
    if top10 is None:
        top10, _ = baseline_retrieve(query)
    reranked, ce = rerank(query, top10)
    ctx = reranked[:TOP_N_CONTEXT]
    return {"reranked": reranked, "ce": ce, "context": ctx, "answer": llm_generate(build_prompt(query, ctx))}

# quick smoke test
_r = pipeline_1("How does the knight move?")
print(_r["context"], "\n", _r["answer"])

[113, 14, 18] 
 A knight moves to one of the squares nearest to that on which it stands but not on the same rank, file, or diagonal.


## 7. Failure-mode stress tests

Each test query comes with a **gold phrase**: a short string taken from the clause that actually answers the question. Every chunk containing that phrase counts as a gold chunk. Because chunks overlap, a clause can occasionally sit in two chunks. The *gold rank* is the best (lowest) rank of any gold chunk.

The queries are grouped by the failure categories from the assignment. I found them by probing about 60 natural questions against the corpus and keeping the ones where the baseline stumbles. Section 8 also keeps the cases where re-ranking does **not** help.

In [8]:
FAILURE_QUERIES = [
    # ---- Keyword trap / distractor: the query shares words with the wrong passages,
    #      while the real answer uses different terminology.
    dict(category="Keyword trap / synonym", article="11.1",
         query="What score does a player receive for a tie?",
         gold="half point"),
    dict(category="Keyword trap / synonym", article="13.6",
         query="Can the arbiter tell me my opponent forgot to press his clock?",
         gold="refrain from informing"),

    # ---- Exceptions & corner cases: a general rule exists, but a specific exception overrides it.
    dict(category="Exception / corner case", article="App. A.2 (vs. 8.1)",
         query="Do I need to write down my moves in a 30 minute game?",
         gold="Players do not need to record"),
    dict(category="Exception / corner case", article="App. B.3.c",
         query="Can I correct an illegal move in blitz after my opponent has already moved?",
         gold="cannot be corrected"),
    dict(category="Exception / corner case", article="App. A.4.a",
         query="In a rapid game, what if the king and queen were placed on the wrong squares at the start?",
         gold="reverse king and queen"),

    # ---- Negation & constraints: "not allowed", "may not", role-specific limits.
    dict(category="Negation / constraint", article="12.10",
         query="Can a player appeal the arbiter's ruling on a draw claim in the quickplay finish?",
         gold="may not appeal"),
    dict(category="Negation / constraint", article="12.2",
         query="Is the player who has the move allowed to go to the restroom?",
         gold="player having the move is not allowed to leave"),
    dict(category="Negation / constraint", article="9.1.b(1)",
         query="Can a draw offer be taken back?",
         gold="cannot be withdrawn"),

    # ---- Other: the vital chunk is buried at ranks 4-10 by the bi-encoder.
    dict(category="Buried at rank 4-10", article="7.4.a",
         query="Is it illegal to leave a pawn on the last rank without replacing it?",
         gold="promotion of a pawn or capturing"),
    dict(category="Buried at rank 4-10", article="13.6",
         query="Can I ask the arbiter how many moves have been played?",
         gold="not indicate the number of moves"),
]

def gold_ids(phrase):
    ids = [c["chunk_id"] for c in chunks if phrase.lower() in c["text"].lower()]
    assert ids, f"gold phrase not found: {phrase!r}"
    return ids

def best_rank(ranked_ids, gold):
    # 1-based rank of the best gold chunk in a ranked list, or None if absent.
    ranks = [ranked_ids.index(g) + 1 for g in gold if g in ranked_ids]
    return min(ranks) if ranks else None

def full_corpus_rank(query, gold):
    # Rank of the gold chunk over the WHOLE corpus by cosine (to see how far it is buried).
    q = bi_encoder.encode([query], normalize_embeddings=True).astype("float32")
    _, ids = index.search(q, index.ntotal)
    return best_rank(ids[0].tolist(), gold)

### 7.1 Rank-shift table (retrieval only)

In [9]:
def analyse_retrieval(qdef):
    gold = gold_ids(qdef["gold"])
    top10, cos = baseline_retrieve(qdef["query"])
    reranked, ce = rerank(qdef["query"], top10)
    rb, rr = best_rank(top10, gold), best_rank(reranked, gold)
    return {**qdef, "gold_chunks": gold, "top10": top10, "cos": cos, "reranked": reranked, "ce": ce,
            "rank_baseline": rb, "rank_reranked": rr,
            "in_top3_baseline": rb is not None and rb <= TOP_N_CONTEXT,
            "in_top3_reranked": rr is not None and rr <= TOP_N_CONTEXT}

fail_results = [analyse_retrieval(q) for q in FAILURE_QUERIES]

def fmt_rank(r): return f"#{r}" if r else ">10"
shift_df = pd.DataFrame([{
    "category": r["category"], "query": r["query"], "gold article": r["article"],
    "gold chunk(s)": r["gold_chunks"],
    "baseline rank": fmt_rank(r["rank_baseline"]), "re-ranked rank": fmt_rank(r["rank_reranked"]),
    "shift": f"{fmt_rank(r['rank_baseline'])} -> {fmt_rank(r['rank_reranked'])}",
    "in top-3 (P1)": r["in_top3_baseline"], "in top-3 (P2)": r["in_top3_reranked"],
} for r in fail_results])
shift_df.to_csv(RESULTS_DIR / "rank_shifts.csv", index=False)
shift_df[["category", "query", "gold article", "shift", "in top-3 (P1)", "in top-3 (P2)"]]

,category,query,gold article,shift,in top-3 (P1),in top-3 (P2)
0,Keyword trap / synonym,What score does a player receive for a tie?,11.1,#7 -> #2,False,True
1,Keyword trap / synonym,Can the arbiter tell me my opponent forgot to press his clock?,13.6,#10 -> #1,False,True
2,Exception / corner case,Do I need to write down my moves in a 30 minute game?,App. A.2 (vs. 8.1),#5 -> #2,False,True
3,Exception / corner case,Can I correct an illegal move in blitz after my opponent has already moved?,App. B.3.c,#5 -> #2,False,True
4,Exception / corner case,"In a rapid game, what if the king and queen were placed on the wrong squares at the start?",App. A.4.a,#3 -> #1,True,True
5,Negation / constraint,Can a player appeal the arbiter's ruling on a draw claim in the quickplay finish?,12.10,#6 -> #2,False,True
6,Negation / constraint,Is the player who has the move allowed to go to the restroom?,12.2,#3 -> #1,True,True
7,Negation / constraint,Can a draw offer be taken back?,9.1.b(1),#3 -> #1,True,True
8,Buried at rank 4-10,Is it illegal to leave a pawn on the last rank without replacing it?,7.4.a,#5 -> #2,False,True
9,Buried at rank 4-10,Can I ask the arbiter how many moves have been played?,13.6,#4 -> #3,False,True


### 7.2 Detailed top-10 view per query
For every query: the 10 bi-encoder candidates with their cosine rank and score, the cross-encoder score, and the new rank. ★ marks the gold chunk(s).

In [10]:
def top10_table(r):
    new_rank = {cid: i + 1 for i, cid in enumerate(r["reranked"])}
    ce_of = dict(zip(r["reranked"], r["ce"]))
    rows = []
    for i, (cid, cs) in enumerate(zip(r["top10"], r["cos"])):
        rows.append({"gold": "★" if cid in r["gold_chunks"] else "", "chunk": cid,
                     "page": chunks[cid]["page"], "bi rank": i + 1, "cosine": round(cs, 3),
                     "CE score": round(ce_of[cid], 2), "CE rank": new_rank[cid],
                     "text": chunk_texts[cid][:110] + "…"})
    return pd.DataFrame(rows)

for r in fail_results:
    print("=" * 110)
    print(f"[{r['category']}]  {r['query']}")
    print(f"gold chunk(s) {r['gold_chunks']} (Art. {r['article']}): rank "
          f"{fmt_rank(r['rank_baseline'])} -> {fmt_rank(r['rank_reranked'])}")
    display(top10_table(r))

[Keyword trap / synonym]  What score does a player receive for a tie?
gold chunk(s) [82, 83] (Art. 11.1): rank #7 -> #2


,gold,chunk,page,bi rank,cosine,CE score,CE rank,text
0,,61,11,1,0.434,-6.75,6,move before making another. Both players must record the offer of a draw on the scoresheet. (See Appendix C.13…
1,,62,12,2,0.433,-5.03,4,"If a player is unable to keep score, an assistant, who must be acceptable to the arbiter, may be provided by t…"
2,,86,14,3,0.426,-4.20,1,"if the opponent cannot win the game by any series of legal moves, his score shall be a draw. c. Smoking is per…"
3,,63,12,4,0.394,-4.82,3,arbiter throughout the game. 8.3 The scoresheets are the property of the organisers of the event. 8.4 If a pla…
4,,68,12,5,0.393,-7.01,8,"conclusion of the game both players shall sign both scoresheets, indicating the result of the game. Even if in…"
5,,65,12,6,0.389,-6.22,5,"both players shall update their scoresheets, using the arbiter's or the opponent's scoresheet. b. If only one …"
6,★,82,14,7,0.387,-4.23,2,"c. If the arbiter has rejected the claim, the opponent shall be awarded two extra minutes time. d. The decisio…"
7,,60,11,8,0.374,-6.76,7,"in the algebraic notation ( See Appendix C ), on the scoresheet prescribed for the competition. It is forbidde…"
8,,4,2,9,0.372,-9.46,10,"or qualifying event, or for a FIDE title or rating tournament. BASIC RULES OF PLAY Article 1: The nature and o…"
9,,67,12,10,0.359,-7.08,9,"and the number of moves made, if this information is available, before reconstruction takes place. 8.6 If the …"


[Keyword trap / synonym]  Can the arbiter tell me my opponent forgot to press his clock?
gold chunk(s) [93] (Art. 13.6): rank #10 -> #1


,gold,chunk,page,bi rank,cosine,CE score,CE rank,text
0,,44,10,1,0.624,-0.63,7,player to perform this operation. His clock shall be adjusted by the arbiter in an equitable way. 6.8 A flag i…
1,,49,10,2,0.602,0.82,3,is not available. c. The arbiter shall decide when the game is to be restarted in either case. d. If a player …
2,,41,9,3,0.593,2.97,2,"he arrives, unless the rules of the com petition specify or the arbiter decides otherwise. 6.7 a. During the g…"
3,,79,13,4,0.587,-1.08,9,"the move, has less than two minutes left on his clock, he may claim a draw before his flag falls. He shall sum…"
4,,43,10,5,0.584,0.07,5,"c. The players must handle the chess clock properly. It is forbidden to punch it forcibly, to pick it up or to…"
5,,46,10,6,0.580,0.71,4,in the absence of any evident defect. A chess clock with an evident de fect shall be replaced. The arbiter sha…
6,,48,10,7,0.567,-0.89,8,"if it happens in the period of a game, in which all remaining moves must be completed. 6.12 a. If the game nee…"
7,,155,25,8,0.561,-0.38,6,"that the time used has been incorrectly indicated on either clock, the error must be corrected. If the error i…"
8,,82,14,9,0.557,-1.87,10,"c. If the arbiter has rejected the claim, the opponent shall be awarded two extra minutes time. d. The decisio…"
9,★,93,15,10,0.556,6.30,1,"the number of moves made, except in applying Article 8.5, when at least one flag has fallen . The arbiter shal…"


[Exception / corner case]  Do I need to write down my moves in a 30 minute game?
gold chunk(s) [95] (Art. App. A.2 (vs. 8.1)): rank #5 -> #2


...

,gold,chunk,page,bi rank,cosine,CE score,CE rank,text
0,,69,12,1,0.651,2.22,2,of the arbiter. b. If the rules of a competition allow a draw agreement the following apply: (1) A player wish…
1,,71,13,2,0.639,1.37,3,(2) The offer of a draw shall be noted by each player on his scoresheet with a symbol. (See Appendix C.13) (3)…
2,★,70,12,3,0.572,3.20,1,is still valid but Article 12.6 must be considered. No conditions can be attached to the offer. In both cases …
3,,61,11,4,0.525,-0.17,4,move before making another. Both players must record the offer of a draw on the scoresheet. (See Appendix C.13…
4,,60,11,5,0.513,-3.45,5,"in the algebraic notation ( See Appendix C ), on the scoresheet prescribed for the competition. It is forbidde…"
5,,68,12,6,0.508,-7.91,9,"conclusion of the game both players shall sign both scoresheets, indicating the result of the game. Even if in…"
6,,76,13,7,0.470,-6.98,7,moves have been made by each player without the movement of any pawn and without any capture. 9.4 If the playe…
7,,77,13,8,0.444,-8.72,10,"to withdraw his claim. a. If the claim is found to be correct, the game is immediately drawn. b. If the claim …"
8,,80,14,9,0.435,-7.86,8,"a. If the arbiter agrees the opponent is making no effort to win the game by normal means, or that it is not p…"
9,,73,13,10,0.430,-6.78,6,"to make this move, or b. has just appeared, and the player claiming the draw has the move. Positions as in (a)…"


[Buried at rank 4-10]  Is it illegal to leave a pawn on the last rank without replacing it?
gold chunk(s) [54] (Art. 7.4.a): rank #5 -> #2


,gold,chunk,page,bi rank,cosine,CE score,CE rank,text
0,,31,8,1,0.492,-3.40,4,"move is not yet made, but the player no longer has the right to play the pawn to another square. The move is c…"
1,,30,8,2,0.491,-1.15,1,"no longer has the right to make any move other than castling on that side, if this is legal c. in the case of …"
2,,74,13,3,0.460,-5.03,7,same if a pawn that could have been captured en passant can no longer be captured in this manner. When a king …
3,,17,6,4,0.452,-2.63,3,e. When a pawn reaches the rank furthest from its starting position it must be exchanged as part of the same m…
4,★,54,11,5,0.444,-1.53,2,for the arbiter's assistance. The arbiter may penalise the player who displaced the pieces. 7.4 a. If during a…
5,,27,7,6,0.401,-3.52,5,player must make anot her legal move with his king (which may include castling on the other side ). If the kin…
6,,83,14,7,0.397,-6.94,8,forfeits scores no points (0) and a player who draws his game scores a half point (½). Article 12: The conduct…
7,,151,25,8,0.392,-9.49,9,shall be handed to the arbiter for safe -keeping and opened on the absent player's arrival. 10. The player sha…
8,,108,18,9,0.387,-5.03,6,placed on the first and second ranks; the black pieces and pawns on the eighth and seventh ranks. C.7 As a con…
9,,34,8,10,0.386,-10.46,10,the opponent's king with any series of legal moves. The game is said to end in a 'dead position'. This immedia…


[Buried at rank 4-10]  Can I ask the arbiter how many moves have been played?
gold chunk(s) [92] (Art. 13.6): rank #4 -> #3


,gold,chunk,page,bi rank,cosine,CE score,CE rank,text
0,,93,15,1,0.651,2.96,2,"the number of moves made, except in applying Article 8.5, when at least one flag has fallen . The arbiter shal…"
1,,75,13,2,0.587,3.24,1,"it is moved. 9.3 The game is drawn, upon a correct claim by the player having the move, if: a. he writes his m…"
2,,66,12,3,0.529,-1.66,7,"player's move, he may use his opponent's scoresheet, but must return it before making a move. c. If no complet…"
3,★,92,15,4,0.526,0.18,3,in the game by the opponent to the maximum available for that game g. expulsion from the event. 13.5 The arbit…
4,,72,13,5,0.517,-1.49,6,"when the same position, for at least the third time (not necessarily by a repetition of moves): a. is about to…"
5,,67,12,6,0.505,-3.66,9,"and the number of moves made, if this information is available, before reconstruction takes place. 8.6 If the …"
6,,35,8,7,0.503,-4.17,10,is about to appear or has appeared on the chessboard at least three times. (See Article 9.2) e. The game may b…
7,,100,17,8,0.503,-0.10,4,is one where all the moves must be made in a fixed time of less than 15 minutes for each player; or the allott…
8,,142,24,9,0.494,-0.89,5,Guidelines in case a game needs to be adjourned 1. a. If a game is not finished at the end of the time prescri…
9,,63,12,10,0.492,-1.69,8,arbiter throughout the game. 8.3 The scoresheets are the property of the organisers of the event. 8.4 If a pla…


### 7.3 Answer 1 (baseline) vs. Answer 2 (re-ranked)
Both answers use the same LLM and prompt. The only difference is which 3 chunks are in the context.

In [11]:
def compare_answers(r):
    ctx1 = r["top10"][:TOP_N_CONTEXT]
    ctx2 = r["reranked"][:TOP_N_CONTEXT]
    a1 = llm_generate(build_prompt(r["query"], ctx1))
    a2 = llm_generate(build_prompt(r["query"], ctx2))
    return {**r, "ctx1": ctx1, "ctx2": ctx2, "answer1": a1, "answer2": a2}

fail_results = [compare_answers(r) for r in fail_results]

def print_comparison(r):
    w = lambda s: textwrap.fill(s, 105, initial_indent="    ", subsequent_indent="    ")
    print("=" * 110)
    print(f"[{r['category']}]  Q: {r['query']}")
    print(f"Gold: Art. {r['article']} -> chunk(s) {r['gold_chunks']}  |  rank "
          f"{fmt_rank(r['rank_baseline'])} -> {fmt_rank(r['rank_reranked'])}")
    print("\nGold clause text:")
    print(w(chunk_texts[r['gold_chunks'][0]]))
    print(f"\nAnswer 1 (baseline, context chunks {r['ctx1']}):")
    print(w(r["answer1"]))
    print(f"\nAnswer 2 (re-ranked, context chunks {r['ctx2']}):")
    print(w(r["answer2"]))
    print()

for r in fail_results:
    print_comparison(r)

[Keyword trap / synonym]  Q: What score does a player receive for a tie?
Gold: Art. 11.1 -> chunk(s) [82, 83]  |  rank #7 -> #2

Gold clause text:
    c. If the arbiter has rejected the claim, the opponent shall be awarded two extra minutes time. d.
    The decision of the arbiter shall be final relating to (a), (b) and (c). Article 11: Points 11.1
    Unless announced otherwise in advance, a player who wins his game , or wins by forfeit, scores one
    point (1), a player who loses his game , or forfeits scores no points (0) and a player who draws his
    game scores a half point (½). Article 12:

Answer 1 (baseline, context chunks [61, 62, 86]):
    A player receives a score of a draw if the opponent cannot win the game by any series of legal moves.
    This is indicated in the passage where it states "if the opponent cannot win the game by any series
    of legal moves, his score shall be a draw."

Answer 2 (re-ranked, context chunks [86, 82, 63]):
    A player receives a half point

## 8. Cases where re-ranking does **not** help
Re-ranking is not a free win. Three typical limitations show up on this document:
1. **Recall ceiling.** If the gold chunk isn't in the bi-encoder's top-10, the cross-encoder never sees it.
2. **Gold chunk demoted.** The cross-encoder (trained on MS MARCO web search) can prefer a chunk that *looks* like a direct answer over the one that actually contains the restriction.
3. **Still outside the top-3.** The re-ranker moves the gold chunk up a little, but not far enough.

In [12]:
LIMITATION_QUERIES = [
    dict(category="Recall ceiling (gold not in top-10)", article="6.12.b",
         query="What should a player do if the piece he wants to promote to is not available?",
         gold="piece required is not available"),
    dict(category="Re-ranker demotes gold", article="3.8.b(2)",
         query="Can a player castle when his king is currently in check?",
         gold="the square on which the king stands"),
    dict(category="Still outside top-3", article="7.3",
         query="What if I accidentally knock pieces off their squares?",
         gold="displaces one or more pieces"),
]

lim_results = [compare_answers(analyse_retrieval(q)) for q in LIMITATION_QUERIES]
for r in lim_results:
    r["full_corpus_rank"] = full_corpus_rank(r["query"], r["gold_chunks"])

lim_df = pd.DataFrame([{
    "category": r["category"], "query": r["query"], "gold article": r["article"],
    "rank in full corpus (cosine)": f"#{r['full_corpus_rank']}",
    "shift (top-10)": f"{fmt_rank(r['rank_baseline'])} -> {fmt_rank(r['rank_reranked'])}",
} for r in lim_results])
lim_df.to_csv(RESULTS_DIR / "limitations.csv", index=False)
display(lim_df)
for r in lim_results:
    display(top10_table(r))
    print_comparison(r)

,category,query,gold article,rank in full corpus (cosine),shift (top-10)
0,Recall ceiling (gold not in top-10),What should a player do if the piece he wants to promote to is not available?,6.12.b,#32,>10 -> >10
1,Re-ranker demotes gold,Can a player castle when his king is currently in check?,3.8.b(2),#5,#5 -> #8
2,Still outside top-3,What if I accidentally knock pieces off their squares?,7.3,#8,#8 -> #7


,gold,chunk,page,bi rank,cosine,CE score,CE rank,text
0,,27,7,1,0.509,3.73,1,player must make anot her legal move with his king (which may include castling on the other side ). If the kin…
1,,30,8,2,0.392,0.68,2,"no longer has the right to make any move other than castling on that side, if this is legal c. in the case of …"
2,,121,20,3,0.389,0.14,3,the black pieces being specially marked. E.2 The following regulations shall govern play: 1. The moves shall b…
3,,87,14,4,0.383,-8.71,5,relevant data. 12.5 Players who have finished their games shall be considered to be spectators. 12.6 It is for…
4,,61,11,5,0.355,-9.69,6,move before making another. Both players must record the offer of a draw on the scoresheet. (See Appendix C.13…
5,,83,14,6,0.353,-7.77,4,forfeits scores no points (0) and a player who draws his game scores a half point (½). Article 12: The conduct…
6,,70,12,7,0.340,-9.93,7,is still valid but Article 12.6 must be considered. No conditions can be attached to the offer. In both cases …
7,,4,2,8,0.336,-10.21,10,"or qualifying event, or for a FIDE title or rating tournament. BASIC RULES OF PLAY Article 1: The nature and o…"
8,,89,15,9,0.334,-10.01,9,case of Article 10.2.d or Appendix D a player may not appeal against the decision of the arbiter. Otherwise a …
9,,129,21,10,0.334,-9.94,8,any or all of the following duties: a. make either player's move on the board of the opponent b. announce the …


[Recall ceiling (gold not in top-10)]  Q: What should a player do if the piece he wants to promote to is not available?
Gold: Art. 6.12.b -> chunk(s) [48]  |  rank >10 -> >10

Gold clause text:
    if it happens in the period of a game, in which all remaining moves must be completed. 6.12 a. If the
    game needs to be interrupted, the arbiter shall stop the clocks. b. A player may stop the clocks only
    in order to seek the arbiter's assistance, for example when promotion has taken place and the piece
    required is not available. c. The arbiter shall decide when the game is to be restarted in either
    case. d.

Answer 1 (baseline, context chunks [27, 30, 121]):
    If the player wants to promote a pawn but the piece he desires (such as a queen) is not available, he
    must choose the next best available piece that can be promoted to, such as a rook, bishop, or knight.
    The player must then announce which piece they have chosen for the promotion.

Answer 2 (re-ranked, context

,gold,chunk,page,bi rank,cosine,CE score,CE rank,text
0,,6,2,1,0.594,1.17,3,"attack, exposing one's own king to attack and also 'capturing' the opponent's king are not allowed. The oppone…"
1,,22,7,2,0.590,0.46,4,or [b] if there is any piece between the king and the rook with which castling is to be effected. 3.9 The king…
2,,26,7,3,0.584,1.59,2,a. deliberately touches his king and rook he must castle on that side if it is legal to do so b. deliberately …
3,,23,7,4,0.555,-1.14,6,will either expose the king of the same colour to check or leave that king in check. Article 4: The act of mov…
4,★,21,7,5,0.553,-3.70,8,"b. (1) The right to castle has been lost: [a] if the king has already moved, or [b] with a rook that has alrea…"
5,,5,2,6,0.535,-1.47,7,"the game. A player is said to ' have the move ', when his opponent's move has been ' made'. (See Article 6.7) …"
6,,45,10,7,0.514,-0.58,5,"not complete the prescribed number of moves in the allotted time, the game is lost by the player. However, the…"
7,,102,18,8,0.510,2.03,1,"opponent cannot checkmate the player's king by any possible series of legal moves, then the claimant is entitl…"
8,,139,23,9,0.500,-4.87,9,"To avoid any misunderstanding, it may be useful to state ""I am about to castle"" before castling. 2. In some st…"
9,,97,17,10,0.472,-5.86,10,setting. In case of reverse king and queen placement castling with this king is not allowed. b. The arbiter sh…


[Re-ranker demotes gold]  Q: Can a player castle when his king is currently in check?
Gold: Art. 3.8.b(2) -> chunk(s) [21]  |  rank #5 -> #8

Gold clause text:
    b. (1) The right to castle has been lost: [a] if the king has already moved, or [b] with a rook that
    has already moved. (2) Castling is prevented temporarily: [a] if the square on which the king stands,
    or the square which it must cross, or the square which it is to occupy, is attacked by one or more of
    the opponent's pieces, or [b] if there is any piece between the king and the rook with which castling
    is to be effected.

Answer 1 (baseline, context chunks [6, 22, 26]):
    No, a player cannot castle when their king is currently in check as per Article 4.3.a of the FIDE
    Laws of Chess, which states that if a player touches both the king and the rook intended for
    castling, and castling is illegal due to the king being in check, the player must make another legal
    move with their king, including cast

,gold,chunk,page,bi rank,cosine,CE score,CE rank,text
0,,11,4,1,0.421,-2.95,3,Article 3: The moves of the pieces 3.1 It is not permitted to move a piece to a square occupied by a piece of …
1,,12,4,2,0.395,-2.33,2,piece if the piece could make a capture on that square according to the Articles 3.2 to 3.8. A piece is consid…
2,,28,8,3,0.369,-3.95,4,"4.5 If none of the pieces touched can be moved or captured, the player may make any legal move. 4.6 When, as a…"
3,,54,11,4,0.350,-9.33,8,for the arbiter's assistance. The arbiter may penalise the player who displaced the pieces. 7.4 a. If during a…
4,,83,14,5,0.328,-11.31,10,forfeits scores no points (0) and a player who draws his game scores a half point (½). Article 12: The conduct…
5,,57,11,6,0.319,-1.85,1,"a third illegal move by the same player, the arbiter shall declare the game lost by this player. However, the …"
6,,25,7,7,0.315,-9.65,9,"or, if this is illegal, move or capture the first piece touched which can be moved or captured. If it is uncle…"
7,★,53,11,8,0.307,-8.20,7,"a game h as begun with colours reversed then it shall continue, unless the arbiter rules otherwise. 7.3 If a p…"
8,,29,8,9,0.302,-6.04,6,"having placed his own piece on its new square, has released this capturing piece from his hand b. in the case …"
9,,15,5,10,0.298,-6.00,5,"it may advance two squares along the same file provided both squares are unoccupied, or c. the pawn may move t…"


[Still outside top-3]  Q: What if I accidentally knock pieces off their squares?
Gold: Art. 7.3 -> chunk(s) [53]  |  rank #8 -> #7

Gold clause text:
    a game h as begun with colours reversed then it shall continue, unless the arbiter rules otherwise.
    7.3 If a player displaces one or more pieces, he shall re -establish the correct position on his own
    time. If necessary, either the player or his opponent shall stop the clocks and ask for the arbiter's
    assistance. The arbiter may penalise the player who displaced the pieces. 7.4 a.

Answer 1 (baseline, context chunks [11, 12, 28]):
    If you accidentally knock pieces off their squares, these pieces are considered to be removed from
    the chessboard as part of the same move according to Article 3.1 of the FIDE Laws of Chess. This
    means they are no longer on the board and cannot be moved back into position.

Answer 2 (re-ranked, context chunks [57, 12, 11]):
    If during a game it is found that pieces have been displa

### Save all answers for the report

In [13]:
def to_record(r):
    return {"category": r["category"], "query": r["query"], "gold_article": r["article"],
            "gold_chunks": r["gold_chunks"], "rank_baseline": r["rank_baseline"],
            "rank_reranked": r["rank_reranked"], "ctx_baseline": r["ctx1"], "ctx_reranked": r["ctx2"],
            "answer1_baseline": r["answer1"], "answer2_reranked": r["answer2"]}

all_records = [to_record(r) for r in fail_results + lim_results]
pd.DataFrame(all_records).to_csv(RESULTS_DIR / "answers.csv", index=False)

with open(RESULTS_DIR / "answers.md", "w", encoding="utf-8") as f:
    f.write(f"# Answer 1 vs Answer 2  (LLM: {LLM_NAME})\n\n")
    for r in fail_results + lim_results:
        f.write(f"## [{r['category']}] {r['query']}\n\n")
        f.write(f"* Gold: Art. {r['article']} (chunk {r['gold_chunks']}), rank "
                f"{fmt_rank(r['rank_baseline'])} -> {fmt_rank(r['rank_reranked'])}\n")
        f.write(f"* Gold text: {chunk_texts[r['gold_chunks'][0]]}\n\n")
        f.write(f"**Answer 1 (baseline, chunks {r['ctx1']}):** {r['answer1']}\n\n")
        f.write(f"**Answer 2 (re-ranked, chunks {r['ctx2']}):** {r['answer2']}\n\n---\n\n")
print("Saved results/answers.csv and results/answers.md")

Saved results/answers.csv and results/answers.md


## 9. Aggregate retrieval metrics on a wider labelled query set
Beyond the hand-picked failures, here are Hit@1, Hit@3 and MRR@10 on a wider set of labelled questions. This set includes many "easy" questions where the baseline is already correct.

*Caveat:* I wrote these questions while hunting for failures, so the set is biased towards hard cases. Treat the numbers as indicative, not as a benchmark.

In [14]:
EVAL_QUERIES = [
    ("What happens if my cell phone rings during a game?", "produces a sound"),
    ("What happens to a player who shows up late for the start of the game?", "arrives at the chessboard after the start"),
    ("What is the penalty for making an illegal move in a standard game?", "for the first two illegal moves"),
    ("When is castling not allowed?", "right to castle has been lost"),
    ("What happens if a player claims a threefold repetition draw but the claim is wrong?", "add three minutes"),
    ("Can I promote a pawn to a second queen while my first queen is still on the board?", "not restricted to pieces"),
    ("What happens if both flags fall and nobody knows which fell first?", "impossible to establish which flag"),
    ("Is a player allowed to leave the playing hall during the game?", "not allowed to leave the 'playing venue'"),
    ("What happens if I touch one of my opponent's pieces?", "must capture the first piece touched"),
    ("Can I take back a move after letting go of the piece?", "released on a square, it"),
    ("Is the player allowed to write down his move before playing it?", "forbidden to write the moves in advance"),
    ("Do I have to keep writing my moves when I am very short on time?", "less than five minutes"),
    ("What happens if the clock was set incorrectly?", "setting of either or both clocks was incorrect"),
    ("Can a player offer a draw while his clock is running?", "offer a draw shall do so after having made a move"),
    ("Is smoking allowed at the board?", "Smoking is permitted only"),
    ("What happens if the game started with the pieces set up wrongly?", "initial position of the pieces was incorrect"),
    ("What punishments can an arbiter give to a player?", "following penalties"),
    ("In blitz, what happens when a player makes an illegal move?", "entitled to claim a win"),
    ("Can a spectator tell a player that his opponent's flag has fallen?", "inform only the arbiter"),
    ("How many points does a player get for a draw?", "half point"),
    ("If a player arrives more than one hour late to resume an adjourned game, does he lose?", "more than one hour late"),
    ("What score does a player receive for a tie?", "half point"),
    ("My opponent keeps talking to me during the game, is that allowed?", "distract or annoy"),
    ("What if I accidentally knock pieces off their squares?", "displaces one or more pieces"),
    ("Can I press the clock with my other hand?", "same hand"),
    ("Can the arbiter tell me my opponent forgot to press his clock?", "refrain from informing"),
    ("Can I keep my phone in my pocket if it is turned off?", "completely switched off"),
    ("If I touch my rook first and then my king, can I still castle?", "rook and then his king"),
    ("In a rapid game, what if the king and queen were placed on the wrong squares at the start?", "reverse king and queen"),
    ("Can castling happen on the very first move in Chess960?", "as early as the first move"),
    ("Do I need to write down my moves in a 30 minute game?", "Players do not need to record"),
    ("Do I need to write down my moves in a 10 minute game?", "Players do not need to record"),
    ("Is it allowed to keep my finger above the clock button?", "hover"),
    ("Can a player be punished for slamming the clock?", "punch it forcibly"),
    ("Can I ask the arbiter how many moves have been played?", "not indicate the number of moves"),
    ("Is it illegal to leave a pawn on the last rank without replacing it?", "promotion of a pawn or capturing"),
    ("Can a draw offer be taken back?", "cannot be withdrawn"),
    ("Is it allowed to make a draw offer with conditions?", "No conditions can be attached"),
    ("Who owns the scoresheets?", "property of the organisers"),
    ("What if both players keep refusing to follow the rules?", "lost by both players"),
    ("Can a player appeal the arbiter's ruling on a draw claim in the quickplay finish?", "may not appeal"),
    ("What should a player do if the piece he wants to promote to is not available?", "piece required is not available"),
    ("Can a player resign before an adjourned game is resumed?", "notifies the arbiter that he resigns"),
    ("Is the player who has the move allowed to go to the restroom?", "player having the move is not allowed to leave"),
    ("Is a mobile phone forbidden for spectators too?", "forbidden for anybody"),
    ("What is the minimum size of a board for visually impaired players?", "20 by 20"),
    ("Can a player claim a draw by repetition if the positions differ only in en passant rights?", "Positions are not the same"),
    ("Does the arbiter announce when a flag falls in a rapid game with little supervision?", "refrain from signalling a flag fall"),
    ("Can a player castle when his king is currently in check?", "the square on which the king stands"),
    ("Can I castle if a square the king passes through is attacked?", "the square which it must"),
    ("What happens if a player makes a third illegal move?", "third illegal move"),
    ("Can I correct an illegal move in blitz after my opponent has already moved?", "cannot be corrected"),
]

def metrics(ranks):
    ranks = [r if r is not None else np.inf for r in ranks]
    return {"Hit@1": np.mean([r <= 1 for r in ranks]), "Hit@3": np.mean([r <= 3 for r in ranks]),
            "Hit@10": np.mean([r <= 10 for r in ranks]),
            "MRR@10": np.mean([1 / r if r <= 10 else 0 for r in ranks])}

rb_all, rr_all = [], []
for q, phrase in EVAL_QUERIES:
    g = gold_ids(phrase)
    top10, _ = baseline_retrieve(q)
    reranked, _ = rerank(q, top10)
    rb_all.append(best_rank(top10, g)); rr_all.append(best_rank(reranked, g))

metrics_df = pd.DataFrame([metrics(rb_all), metrics(rr_all)],
                          index=["Pipeline 1: bi-encoder", "Pipeline 2: + cross-encoder"]).round(3)
metrics_df.to_csv(RESULTS_DIR / "retrieval_metrics.csv")
print(f"{len(EVAL_QUERIES)} labelled queries")
display(metrics_df)

moves = pd.Series([("improved" if (b or 99) > (r or 99) else "worse" if (b or 99) < (r or 99) else "same")
                   for b, r in zip(rb_all, rr_all)]).value_counts()
print("Gold-rank change after re-ranking:\n", moves.to_string())

52 labelled queries


,Hit@1,Hit@3,Hit@10,MRR@10
Pipeline 1: bi-encoder,0.462,0.731,0.962,0.611
Pipeline 2: + cross-encoder,0.577,0.923,0.962,0.736


Gold-rank change after re-ranking:
 same        25
improved    21
worse        6


## 10. Latency profiling
Timings are measured with `time.perf_counter()` after a warm-up (the first GPU calls include CUDA initialisation). Each stage is repeated `N_RUNS` times over all failure queries.
* **Baseline retrieval**: embed the query, then FAISS top-10 search.
* **Re-ranking pipeline (retrieval)**: baseline retrieval plus cross-encoder scoring of the 10 pairs and re-sorting.
* **LLM generation** is timed separately because it dominates end-to-end latency and is the same in both pipelines.

In [15]:
N_RUNS = 20
lat_queries = [q["query"] for q in FAILURE_QUERIES]

# warm-up
for q in lat_queries[:3]:
    ids, _ = baseline_retrieve(q); rerank(q, ids)

def time_ms(fn, *a):
    t0 = time.perf_counter(); out = fn(*a); return out, (time.perf_counter() - t0) * 1000

t_embed_search, t_rerank = [], []
for _ in range(N_RUNS):
    for q in lat_queries:
        (ids, _), t1 = time_ms(baseline_retrieve, q)
        _, t2 = time_ms(rerank, q, ids)
        t_embed_search.append(t1); t_rerank.append(t2)

t_embed_search, t_rerank = np.array(t_embed_search), np.array(t_rerank)
t_pipeline2 = t_embed_search + t_rerank

# LLM generation time (identical prompt length class for both pipelines) - fewer runs, it is slow
t_llm = []
for r in fail_results[:5]:
    _, t = time_ms(llm_generate, build_prompt(r["query"], r["ctx1"])); t_llm.append(t)
    _, t = time_ms(llm_generate, build_prompt(r["query"], r["ctx2"])); t_llm.append(t)
t_llm = np.array(t_llm)

def stats(name, arr):
    return {"stage": name, "mean (ms)": arr.mean(), "median (ms)": np.median(arr),
            "p95 (ms)": np.percentile(arr, 95), "n": len(arr)}

latency_df = pd.DataFrame([
    stats("P1 baseline retrieval (embed + FAISS top-10)", t_embed_search),
    stats("Cross-encoder re-rank of 10 pairs (only)", t_rerank),
    stats("P2 re-ranking retrieval (embed + FAISS + CE)", t_pipeline2),
    stats(f"LLM generation ({LLM_NAME})", t_llm),
    stats("P1 end-to-end (retrieval + LLM, est.)", t_embed_search.mean() + t_llm),
    stats("P2 end-to-end (retrieval + CE + LLM, est.)", t_pipeline2.mean() + t_llm),
]).round(2)
latency_df.to_csv(RESULTS_DIR / "latency.csv", index=False)
print(f"Retrieval overhead of re-ranking: +{t_rerank.mean():.1f} ms per query "
      f"({t_pipeline2.mean() / t_embed_search.mean():.1f}x the baseline retrieval time)")
latency_df

Retrieval overhead of re-ranking: +11.2 ms per query (2.7x the baseline retrieval time)


,stage,mean (ms),median (ms),p95 (ms),n
0,P1 baseline retrieval (embed + FAISS top-10),6.42,6.36,6.85,200
1,Cross-encoder re-rank of 10 pairs (only),11.20,10.95,14.31,200
2,P2 re-ranking retrieval (embed + FAISS + CE),17.62,17.32,20.70,200
3,LLM generation (Qwen/Qwen2.5-3B-Instruct),3325.48,2911.33,4960.73,10
4,"P1 end-to-end (retrieval + LLM, est.)",3331.90,2917.75,4967.15,10
5,"P2 end-to-end (retrieval + CE + LLM, est.)",3343.10,2928.95,4978.35,10


## 11. Scaling experiment: why not run the cross-encoder over everything?
A bi-encoder pays its expensive cost **once, offline**, by embedding the corpus. Each query then needs one short encoder pass plus a vector search, which is sub-linear with ANN indexes such as HNSW or IVF. A cross-encoder can't cache anything, because every (query, chunk) pair needs its own full transformer pass. Cost therefore grows linearly with corpus size, *per query*.

Below I time the cross-encoder on 10, 50 and all 156 chunks, then extrapolate the measured per-pair cost to larger corpora.

In [16]:
q = FAILURE_QUERIES[0]["query"]
scale_rows = []
for n in [10, 50, len(chunks)]:
    ids = list(range(n))
    times = [time_ms(rerank, q, ids)[1] for _ in range(10)]
    scale_rows.append({"pairs scored": n, "CE time (ms)": np.mean(times), "ms per pair": np.mean(times) / n})
_, t_bi = time_ms(baseline_retrieve, q)
scale_df = pd.DataFrame(scale_rows).round(3)
display(scale_df)

per_pair = scale_df["ms per pair"].iloc[-1]
extrap = pd.DataFrame([{"corpus size (chunks)": n,
                        "CE over whole corpus, per query": f"{n * per_pair / 1000:,.1f} s",
                        "bi-encoder + CE on top-10, per query": f"~{t_bi + 10 * per_pair:.0f} ms (+ ANN search)"}
                       for n in [156, 10_000, 1_000_000, 100_000_000]])
extrap.to_csv(RESULTS_DIR / "scaling.csv", index=False)
extrap

,pairs scored,CE time (ms),ms per pair
0,10,12.923,1.292
1,50,65.725,1.315
2,156,168.183,1.078


,corpus size (chunks),"CE over whole corpus, per query","bi-encoder + CE on top-10, per query"
0,156,0.2 s,~22 ms (+ ANN search)
1,10000,10.8 s,~22 ms (+ ANN search)
2,1000000,"1,078.0 s",~22 ms (+ ANN search)
3,100000000,"107,800.0 s",~22 ms (+ ANN search)


## Summary of outputs
All tables used in the report are saved to `results/`:
* `rank_shifts.csv`: gold-chunk rank before and after re-ranking for the failure queries
* `answers.md` / `answers.csv`: Answer 1 vs Answer 2 with the chunks each pipeline used
* `limitations.csv`: cases where re-ranking does not help
* `retrieval_metrics.csv`: Hit@k and MRR on the wider query set
* `latency.csv`, `scaling.csv`: latency profiling and scaling extrapolation